<a href="https://colab.research.google.com/github/Yernar8121/Gold-Price-ML-Project/blob/main/preprocessing_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/XAU_1d_data.csv", sep=";")

print(df.head())
print(df.info())
print(df.isnull().sum())

df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M")

df = df.sort_values("Date").reset_index(drop=True)

price_cols = ["Open", "High", "Low", "Close", "Volume"]

for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

df = df.drop_duplicates()

df["Price_Range"] = df["High"] - df["Low"]
df["Open_Close_Diff"] = df["Close"] - df["Open"]
df["High_Low_Ratio"] = df["High"] / df["Low"]

df["Daily_Return"] = df["Close"].pct_change()

df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_3"] = df["Close"].shift(3)
df["Close_Lag_7"] = df["Close"].shift(7)

df["MA_7"] = df["Close"].rolling(window=7).mean()
df["MA_14"] = df["Close"].rolling(window=14).mean()
df["MA_30"] = df["Close"].rolling(window=30).mean()

df["Volatility_7"] = df["Daily_Return"].rolling(window=7).std()
df["Volatility_14"] = df["Daily_Return"].rolling(window=14).std()

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek

df["Target_Next_Close"] = df["Close"].shift(-1)

df = df.dropna().reset_index(drop=True)

print(df.head())
print(df.shape)
print(df.info())

df.to_csv("XAU_processed_dataset.csv", index=False)

               Date   Open   High    Low  Close  Volume
0  2004.06.11 00:00  384.0  384.8  382.8  384.1     272
1  2004.06.14 00:00  384.3  385.8  381.8  382.8    1902
2  2004.06.15 00:00  382.8  388.8  381.1  388.6    1951
3  2004.06.16 00:00  387.1  389.8  382.6  383.8    2014
4  2004.06.17 00:00  383.6  389.3  383.0  387.6    1568
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5531 entries, 0 to 5530
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    5531 non-null   object 
 1   Open    5531 non-null   float64
 2   High    5531 non-null   float64
 3   Low     5531 non-null   float64
 4   Close   5531 non-null   float64
 5   Volume  5531 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 259.4+ KB
None
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
        Date   Open   High    Low  Close  Volume  Price_Range  \
0 2004-07-23  394.3  395.0  386.5  389.3   

In [3]:
features = [
    "Open", "High", "Low", "Close", "Volume",
    "Price_Range", "Open_Close_Diff", "High_Low_Ratio",
    "Daily_Return",
    "Close_Lag_1", "Close_Lag_3", "Close_Lag_7",
    "MA_7", "MA_14", "MA_30",
    "Volatility_7", "Volatility_14",
    "Year", "Month", "Day", "DayOfWeek"
]

X = df[features]
y = df["Target_Next_Close"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

print(X_train.shape, X_test.shape)

(4400, 21) (1101, 21)


In [5]:
df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M")

df = df.sort_values("Date").reset_index(drop=True)

for col in ["Open", "High", "Low", "Close", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()
df = df.drop_duplicates()